# Power System Operations Workshop

This workshop was created for the workshop **Powering the Energy Transition: Different Pieces, One Shared Puzzle** on 2026-10-08 at the Bright Cubes office.

## Premise

Bright Cubes wants to get an additional NVidia DGX Spark and therefore would like to increase their energy contract.

You are a power system operator with only one goal: bringing the energy transition to a conclusion without blackouts.
You are tasked with answering the following question: can we increase the capacity for Bright Cubes?
A customer can only be safely added if these constraints remain satisfied:

1. Given a certain prior simulation of production and consumption, no cable or transformer is allowed to exceed its set capacity. This is called a timeseries analysis.
2. If a cable breaks or is disconnected for maintenance, then there is always a different configuration of the grid available that can run the entire time series without exceeding the equipment capacities. This is called an N-1 analysis.

## Set-up

In [17]:
import power_grid_model as pgm
import power_grid_model_ds as pgm_ds
import power_grid_model_ds.visualizer as pgm_viz

## Grid dataset

The example network is stored in `data/workshop_grid.json`. It has a 150 kV source bus, a single-loop 10.5 kV medium-voltage backbone, two compact 400 V low-voltage feeder groups, and ten symmetric loads. Dual transformer feeds at the source and low-voltage feeders preserve alternate paths for the N-1 analysis exercise.

In [21]:
from pathlib import Path

from power_grid_model.utils import json_deserialize_from_file

input_data = json_deserialize_from_file(Path("data/workshop_grid.json"))
grid = pgm_ds.PowerGridModelInterface(input_data=input_data).create_grid_from_input_data()
grid.check_ids()

print(
    f"Loaded {len(grid.node)} nodes, {len(grid.line)} lines, "
    f"{len(grid.transformer)} transformers, {len(grid.source)} source, "
    f"and {len(grid.sym_load)} symmetric loads."
)

Loaded 23 nodes, 21 lines, 6 transformers, 1 source, and 10 symmetric loads.


### Visualize the network

In [19]:
pgm_viz.visualize(grid)

## Validate that all data is set correctly

In [22]:
from power_grid_model.validation import errors_to_string, validate_input_data

validation_errors = validate_input_data(
    input_data, calculation_type=pgm.CalculationType.power_flow, symmetric=True
)
if validation_errors:
    raise ValueError(errors_to_string(validation_errors, name="input_data", details=True))
print("Input data is valid for a symmetric power flow calculation.")

Input data is valid for a symmetric power flow calculation.


## Validate that no cables are overloaded in the "base scenario"

In [23]:
import numpy as np

model = pgm.PowerGridModel(input_data)
output_data = model.calculate_power_flow(symmetric=True)

for component in (pgm.ComponentType.line, pgm.ComponentType.transformer):
    result = output_data[component]
    overloaded = result["loading"] > 1.0
    print(
        f"{component.value}: max loading {np.max(result['loading']):.1%} "
        f"(id {result['id'][np.argmax(result['loading'])]}), "
        f"overloaded ids: {result['id'][overloaded].tolist() or 'none'}"
    )

line: max loading 77.9% (id 1027), overloaded ids: none
transformer: max loading 65.8% (id 2005), overloaded ids: none
